# Lezione 11 - Protocollo Agent-to-Agent (A2A)


## Configurazione


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv

In [ ]:
import os
import dotenv
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Cos'è il Protocollo A2A?

Il **protocollo Agent-to-Agent (A2A)** è uno standard aperto che permette agli agenti AI di comunicare,
scoprirsi tra loro e collaborare — anche quando sono costruiti su framework diversi o ospitati
da servizi differenti.

Concetti chiave:

- **Scoperta** – Gli agenti pubblicano una *Scheda Agente* che descrive le loro capacità, facilitando
  ad altri agenti (o orchestratori) di trovare lo specialista giusto per un compito.
- **Passaggio di Messaggi** – Gli agenti scambiano messaggi strutturati tramite un protocollo comune, così che una
  richiesta da un agente possa essere compresa e soddisfatta da un altro a prescindere dalla sua implementazione
  interna.
- **Ciclo di Vita del Compito** – A2A definisce stati come *inviato*, *in lavorazione*, *completato* e
  *fallito*, offrendo all’orchestratore piena visibilità su come sta procedendo un compito delegato.

In questa lezione simuleremo una collaborazione in stile A2A collegando tre agenti di viaggio specializzati
in un flusso di lavoro dove ciascun agente contribuisce con la sua competenza e passa i risultati al successivo.


## Creazione di Agenti di Viaggio Specializzati


In [ ]:
currency_agent = client.as_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = client.as_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = client.as_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## Collaborazione Multi-Agente tramite Workflow

Connettiamo i tre agenti in un workflow sequenziale che riflette il passaggio di messaggi A2A:

1. **CurrencyExchangeAgent** riceve la richiesta dell'utente e produce indicazioni sulla valuta.
2. **ActivityPlannerAgent** riceve il contesto arricchito e aggiunge raccomandazioni sulle attività.
3. **TravelManagerAgent** sintetizza entrambi gli input in un breve riepilogo di viaggio finale.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Comprendere A2A in Produzione

In un ambiente di produzione il protocollo A2A sblocca potenti scenari cross-servizio:

| Capacità | Descrizione |
|---|---|
| **Interop cross-framework** | Un agente costruito con un framework può delegare compiti a un agente costruito con qualsiasi altro framework compatibile A2A, permettendo una vera interoperabilità tra organizzazioni. |
| **Confini di servizio** | Gli agenti possono risiedere in microservizi separati, regioni cloud diverse o addirittura organizzazioni differenti pur collaborando senza soluzione di continuità. |
| **Scoperta dinamica** | Un orchestratore può interrogare un registro di Agent Card a runtime per trovare lo specialista più adatto per un dato sotto-compito. |
| **Streaming e notifiche push** | A2A supporta Server-Sent Events (SSE) per aggiornamenti in tempo reale sui progressi e notifiche push per compiti a lunga durata. |

Il flusso di lavoro che abbiamo costruito sopra è una versione semplificata, in-process, di questo schema. In un reale
deployment ogni agente esporrebbe un endpoint HTTP, pubblicherà un Agent Card, e comunicherà
tramite il protocollo A2A JSON-RPC.


## Riepilogo

In questa lezione hai imparato:

1. **Cos'è il protocollo A2A** — uno standard aperto per la scoperta, la messaggistica,
   e la gestione delle attività tra agenti.
2. **Come creare agenti specializzati** — un agente per lo scambio di valute, un agente pianificatore di attività,
   e un orchestratore Travel Manager.
3. **Come collegare gli agenti in un flusso di lavoro** — usando `WorkflowBuilder` per modellare il passaggio
   sequenziale di messaggi tra agenti.
4. **Come funziona A2A in produzione** — permettendo la collaborazione cross-framework, cross-service
   con scoperta dinamica e aggiornamenti in streaming.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Questo documento è stato tradotto utilizzando il servizio di traduzione AI [Co-op Translator](https://github.com/Azure/co-op-translator). Sebbene ci impegniamo per garantire la precisione, si prega di notare che le traduzioni automatizzate possono contenere errori o imprecisioni. Il documento originale nella sua lingua nativa deve essere considerato la fonte autorevole. Per informazioni critiche, si raccomanda una traduzione professionale effettuata da un essere umano. Non siamo responsabili per eventuali malintesi o interpretazioni errate derivanti dall’uso di questa traduzione.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
